In [1]:
# %% [markdown]
# # 🤖 AI SDR Agent — Interactive Demo
# ## Lead Qualification, Personalized Emails & CRM Updates
#
# Following the Ed Donner Agentic AI Course (Week 2, Day 2) pattern.
# Using Groq (Llama 3.3 70B) as the LLM backend.

# %%
# Setup
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv(override=True)

import os
os.environ["OPENAI_API_KEY"] = os.environ.get("GROQ_API_KEY", "")
os.environ["OPENAI_BASE_URL"] = "https://api.groq.com/openai/v1"

from agents import Agent, Runner, trace, function_tool
from src.config import ICP, PRIMARY_MODEL
from src.database import init_database
from src.lead_loader import load_leads_from_csv
from src.models import Lead
import asyncio

print("✅ Setup complete!")


✅ Setup complete!


In [4]:

# %%
# Initialize database
init_database()
print("✅ Database initialized!")

# %%
# Load sample leads
leads = load_leads_from_csv("sample_leads.csv")
print(f"✅ Loaded {len(leads)} leads")
for lead in leads[:5]:
    print(f"  - {lead.full_name} | {lead.job_title} @ {lead.company_name}")





✅ Database initialized!
✅ Loaded 15 leads
  - Sarah Chen | VP of Engineering @ TechFlow Solutions
  - James Rodriguez | Head of Operations @ RetailNova
  - Priya Sharma | Chief Technology Officer @ FinEdge AI
  - Michael O'Brien | IT Director @ GlobalHealth Partners
  - Aisha Patel | Founder & CEO @ Nimbus Dev Studio


In [5]:



# %%
# Step 2: Agent Workflow — Multiple email writers (Ed Donner pattern)

email_writer1 = Agent(
    name="Professional Writer",
    instructions=f"You write professional cold emails for ComplAI. {ICP['company_description']} Keep under 150 words.",
    model=PRIMARY_MODEL,
)

email_writer2 = Agent(
    name="Engaging Writer",
    instructions=f"You write witty, engaging cold emails for ComplAI. {ICP['company_description']} Keep under 150 words.",
    model=PRIMARY_MODEL,
)

email_writer3 = Agent(
    name="Concise Writer",
    instructions=f"You write very concise cold emails for ComplAI. {ICP['company_description']} Keep under 80 words.",
    model=PRIMARY_MODEL,
)

message = f"Write a cold sales email for: {lead.full_name}, {lead.job_title} at {lead.company_name} ({lead.industry}). Notes: {lead.notes}"

with trace("Parallel email generation"):
    results = await asyncio.gather(
        Runner.run(email_writer1, message),
        Runner.run(email_writer2, message),
        Runner.run(email_writer3, message),
    )

outputs = [r.final_output for r in results]
for i, output in enumerate(outputs, 1):
    print(f"\n{'='*50}")
    print(f"EMAIL DRAFT {i}:")
    print(f"{'='*50}")
    print(output)




EMAIL DRAFT 1:
Subject: Streamline SOC2 Compliance for Your Enterprise Clients

Dear Aisha,

As the founder of Nimbus Dev Studio, you understand the importance of SOC2 compliance for your enterprise clients. However, preparing for audits can be time-consuming and labor-intensive.

ComplAI can help. Our AI-powered SaaS platform automates SOC2 compliance and audit preparation, reducing manual work by 80% and getting your clients audit-ready in weeks, not months.

I'd love to discuss how ComplAI can support your agency's growth and help you deliver more value to your clients. Are you available for a quick call?

Best,
[Your Name]

EMAIL DRAFT 2:
Subject: Streamline SOC2 Compliance for Your Enterprise Clients

Hi Aisha,

As the founder of Nimbus Dev Studio, you understand the importance of SOC2 compliance for your enterprise clients. However, preparing for audits can be a tedious and time-consuming process.

At ComplAI, we're revolutionizing SOC2 compliance with AI-powered automation. Our

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: gsk_KhVv********************************************ZLbx. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}
[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: gsk_KhVv********************************************ZLbx. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}
[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Incorrect API key provided: gsk_KhVv********************************************ZLbx. You can find your API key at https://platform.openai.com/account/api-keys.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_api_key"
  }
}
[non-fatal] Tracing client e

In [6]:

# Step 3: Pick the best email 

email_picker = Agent(
    name="Email Picker",
    instructions="Pick the best cold email. Reply with ONLY the selected email, no explanation.",
    model=PRIMARY_MODEL,
)

all_emails = "Email options:\n\n" + "\n\n---EMAIL---\n\n".join(outputs)

with trace("Email selection"):
    best = await Runner.run(email_picker, all_emails)

print("🏆 BEST EMAIL SELECTED:")
print(best.final_output)

# %%
# Step 4: Run the full pipeline on a few leads
from src.pipeline import run_full_pipeline

results = await run_full_pipeline(
    csv_path="../data/sample_leads.csv",
    send_email=False,        # Set to True to actually send emails
    send_notification=False,  # Set to True for Pushover
    max_leads=3,              # Process just 3 leads for demo
)


🏆 BEST EMAIL SELECTED:
Subject: Streamline SOC2 Compliance for Your Enterprise Clients

Dear Aisha,

As the founder of Nimbus Dev Studio, you understand the importance of SOC2 compliance for your enterprise clients. However, preparing for audits can be time-consuming and labor-intensive.

ComplAI can help. Our AI-powered SaaS platform automates SOC2 compliance and audit preparation, reducing manual work by 80% and getting your clients audit-ready in weeks, not months.

I'd love to discuss how ComplAI can support your agency's growth and help you deliver more value to your clients. Are you available for a quick call?

Best,
[Your Name]
🚀 AI SDR Agent Pipeline Starting...
📂 Loading leads from: ../data/sample_leads.csv
📊 Loaded 3 leads
📧 Email sending: OFF
🔔 Push notifications: OFF

Processing lead 1/3: Sarah Chen @ TechFlow Solutions
  ✅ Score: 90/100 | Tier: HOT
  📝 TechFlow Solutions is a strong ICP match, with a company size and industry that align with our targe...
  ✉️  Email: Strea

Error getting response: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=research_lead{"input": "James Rodriguez, Head of Operations at RetailNova, https://retailnova.com, https://linkedin.com/in/jamesrodriguez, 450 employees, E-commerce / Retail, downloaded whitepaper on inventory automation"}</function>\n'}}. (request_id: req_01kmys9chkfresx23b0zh3hzek)


  ❌ Error: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=research_lead{"input": "James Rodriguez, Head of Operations at RetailNova, https://retailnova.com, https://linkedin.com/in/jamesrodriguez, 450 employees, E-commerce / Retail, downloaded whitepaper on inventory automation"}</function>\n'}}
  ⏳ Waiting 3 seconds (rate limit)...

Processing lead 3/3: Priya Sharma @ FinEdge AI


Error getting response: Error code: 400 - {'error': {'message': 'tool call validation failed: attempted to call tool \'research_lead,{"input": "Priya Sharma, Chief Technology Officer, FinEdge AI, FinTech / AI, 85 employees, https://finedgeai.com, https://linkedin.com/in/priyasharma, LinkedIn Outbound, Commented on our CEO\'s LinkedIn post about AI compliance"}\' which was not in request.tools', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=research_lead,{"input": "Priya Sharma, Chief Technology Officer, FinEdge AI, FinTech / AI, 85 employees, https://finedgeai.com, https://linkedin.com/in/priyasharma, LinkedIn Outbound, Commented on our CEO\'s LinkedIn post about AI compliance"}></function>'}}. (request_id: req_01kmys9me4ejs8m5majcgr2sfy)


  ❌ Error: Error code: 400 - {'error': {'message': 'tool call validation failed: attempted to call tool \'research_lead,{"input": "Priya Sharma, Chief Technology Officer, FinEdge AI, FinTech / AI, 85 employees, https://finedgeai.com, https://linkedin.com/in/priyasharma, LinkedIn Outbound, Commented on our CEO\'s LinkedIn post about AI compliance"}\' which was not in request.tools', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=research_lead,{"input": "Priya Sharma, Chief Technology Officer, FinEdge AI, FinTech / AI, 85 employees, https://finedgeai.com, https://linkedin.com/in/priyasharma, LinkedIn Outbound, Commented on our CEO\'s LinkedIn post about AI compliance"}></function>'}}

📊 PIPELINE SUMMARY
Total Leads Processed: 3
🔥 Hot:  1
🟡 Warm: 0
🔵 Cold: 2
📈 Average Score: 30.0/100


In [7]:

# %%
# Step 5: View results
for r in sorted(results, key=lambda x: x.score, reverse=True):
    emoji = {"hot": "🔥", "warm": "🟡", "cold": "🔵"}.get(r.tier, "")
    print(f"\n{emoji} {r.lead_id} — Score: {r.score}/100 ({r.tier.upper()})")
    print(f"  Reasoning: {r.reasoning[:120]}...")
    if r.personalized_email_subject:
        print(f"  Email Subject: {r.personalized_email_subject}")
    print(f"  Next Action: {r.next_best_action}")

# %%
# Step 6: Check database stats
from src.database import get_dashboard_stats
stats = get_dashboard_stats()
print("\n📊 CRM Dashboard Stats:")
for k, v in stats.items():
    print(f"  {k}: {v}")

print("\n✅ Demo complete! Run 'python dashboard/app.py' for the full dashboard.")


🔥 L001 — Score: 90/100 (HOT)
  Reasoning: TechFlow Solutions is a strong ICP match, with a company size and industry that align with our target market, and a clea...
  Email Subject: Streamline SOC2 Compliance for TechFlow Solutions
  Next Action: Schedule a demo to showcase ComplAI's capabilities in automating SOC2 compliance and audit preparation.

🔵 L002 — Score: 0/100 (COLD)
  Reasoning: Error during processing: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. ...
  Next Action: Retry processing

🔵 L003 — Score: 0/100 (COLD)
  Reasoning: Error during processing: Error code: 400 - {'error': {'message': 'tool call validation failed: attempted to call tool \'...
  Next Action: Retry processing

📊 CRM Dashboard Stats:
  total_leads: 15
  total_qualified: 3
  hot: 1
  warm: 0
  cold: 2
  avg_score: 30.0
  emails_sent: 0

✅ Demo complete! Run 'python dashboard/app.py' for the full dashboard.
